# Results for model: mistralai/mistral-large-3-675b-instruct-2512

In [1]:
import polars as pl
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

data_path = './jane_street_data/train.parquet'

data = pl.read_parquet(data_path)

def calculate_top_quantile(data, feature_col, target_col, date_col, quantile=0.15):
    top_quantile_values = (
        data
        .group_by(date_col)
        .agg(
            pl.col(feature_col).quantile(1 - quantile).alias(f'{feature_col}_top_quantile')
        )
    )
    return (
        data
        .join(top_quantile_values, on=date_col)
        .with_columns(
            pl.when(pl.col(feature_col) >= pl.col(f'{feature_col}_top_quantile'))
            .then(1)
            .otherwise(0)
            .alias(f'{feature_col}_top_quantile_flag')
        )
    )

data = calculate_top_quantile(data, 'feature_00', 'responder_6', 'date_id', quantile=0.15)

feature_cols = [col for col in data.columns if col.startswith('feature_') or col.endswith('_top_quantile_flag')]
target_col = 'responder_6'

X = data.select(feature_cols).to_numpy()
y = data.select(target_col).to_numpy().ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist'
)

model.fit(X_train, y_train)

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet